# Learn RAG with Ollama

Retrieval-Augmented Generation (RAG) helps a language model answer questions using information you provide at question time. Instead of expecting the model to remember your documents, the app finds relevant passages and sends them along with the question.

In this notebook, we will build the flow in small steps:

1. Prepare and split a few documents into chunks.
2. Ask Ollama to turn each chunk into an embedding (a list of numbers representing meaning).
3. Embed a question and find the most similar chunks.
4. Give those chunks to an Ollama language model and generate a grounded answer.

We keep vectors in Python memory so the retrieval process is easy to inspect. A vector database such as Qdrant can replace that part when you need persistence or a larger collection.

## Before you start

Install and start [Ollama](https://ollama.com/), then pull the models used by this project. Run these commands in a terminal, not in a notebook cell:

```bash
ollama pull nomic-embed-text:latest
ollama pull llama3.2:3b
```

The embedding model does semantic search; the language model writes the final answer. This notebook uses the Ollama HTTP API at `http://localhost:11434`. It also uses `langchain-text-splitters`, which is already listed in `rag-demo/requirements.txt`. If that import is unavailable in your notebook environment, install the demo requirements from the `rag-demo` folder.

In [1]:
import json
import math
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

from langchain_text_splitters import RecursiveCharacterTextSplitter

OLLAMA_URL = "http://localhost:11434"
EMBEDDING_MODEL = "nomic-embed-text:latest"
LLM_MODEL = "llama3.2:3b"

print(f"EMBEDDING_MODEL: {EMBEDDING_MODEL}")
print(f"LLM_MODEL: {LLM_MODEL}")

EMBEDDING_MODEL: nomic-embed-text:latest
LLM_MODEL: llama3.2:3b


In [6]:
def ollama_request(endpoint, payload=None):
    """To Send a JSON request to the local ollama server."""
    url = f"{OLLAMA_URL}{endpoint}"
    if payload is None:
        request = Request(url)
    else:
        body = json.dumps(payload).encode("utf-8")
        request = Request(
            url,
            data=body,
            headers={"Content-Type": "application/json"},
            method="POST",
        )
    with urlopen(request, timeout=120) as response:
        return json.loads(response.read().decode("utf-8"))

try:
    installed_models = ollama_request("/api/tags")["models"]
    installed_names = {model['name'] for model in installed_models}
    print("Ollama is responding...")
    for model_name in (EMBEDDING_MODEL, LLM_MODEL):
        if model_name in installed_names:
            print(f"Found {model_name}")
        else:
            print(f"Missing model {model_name}")
except (URLError, TimeoutError, KeyError) as error:
    print("Could not reach ollama")
    print(f"details: {error}")

Ollama is responding...
Found nomic-embed-text:latest
Found llama3.2:3b


## 1. Prepare the knowledge base

These example notes are our source of truth. The model will only see the passages that retrieval selects later. Each document has a source label so we can trace an answer back to its evidence. Replace these examples with text from your own files when you are ready.

In [7]:
documents = [
    {
        "source": "garden-guide.txt",
        "text": (
            "Tomatoes grow best in a sunny location that receives at least six hours of direct light each day. "
            "Water the soil deeply when the top few centimetres feel dry. Water near the soil rather than over the leaves. "
            "A layer of mulch helps the soil retain moisture and reduces weeds."
        ),
    },
    {
        "source": "garden-guide.txt",
        "text": (
            "Basil prefers warm weather and well-drained soil. Pinch off the growing tips to encourage a fuller plant. "
            "Harvest leaves in the morning for the strongest aroma. Basil is sensitive to cold and should be protected "
            "when nighttime temperatures fall."
        ),
    },
    {
        "source": "compost-notes.txt",
        "text": (
            "A healthy compost pile needs a balance of green materials, such as vegetable scraps, and brown materials, "
            "such as dry leaves or shredded cardboard. Keep the pile damp like a wrung-out sponge and turn it regularly "
            "to add oxygen. Finished compost is dark, crumbly, and smells earthy."
        ),
    },
    {
        "source": "pollinator-notes.txt",
        "text": (
            "Plant flowers with different bloom times to provide nectar throughout the growing season. Native plants are "
            "often well suited to local pollinators. Avoid spraying pesticides on open flowers, where bees and other "
            "pollinators may be feeding. A shallow dish of water with stones gives insects a place to land."
        ),
    },
]

print(f"Knowledge base: {len(documents)} documents")
for document in documents:
    print(f"{document['source']}: {len(document['text'])} characters")

Knowledge base: 4 documents
garden-guide.txt: 274 characters
garden-guide.txt: 245 characters
compost-notes.txt: 281 characters
pollinator-notes.txt: 306 characters


## 2. Split documents into chunks

Embedding a whole book as one item makes it hard to retrieve just the useful passage. Chunking creates smaller pieces for search. A little overlap between neighboring chunks helps avoid splitting an idea at a boundary. We keep each chunk's source metadata so we can trace results back to their documents.

In [8]:
chunk_size = 240
chunk_overlap = 40
splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
)

chunk_documents = splitter.create_documents(
    texts=[document["text"] for document in documents],
    metadatas=[{"source": document['source']} for document in documents]
)
print(f"Created {len(chunk_documents)} chunk")
for index, chunk in enumerate(chunk_documents, start=1):
    print(f"\nChunk {index} | Source: {chunk.metadata['source']} | {len(chunk.page_content)} characters")
    print(chunk.page_content)

Created 8 chunk

Chunk 1 | Source: garden-guide.txt | 239 characters
Tomatoes grow best in a sunny location that receives at least six hours of direct light each day. Water the soil deeply when the top few centimetres feel dry. Water near the soil rather than over the leaves. A layer of mulch helps the soil

Chunk 2 | Source: garden-guide.txt | 74 characters
leaves. A layer of mulch helps the soil retain moisture and reduces weeds.

Chunk 3 | Source: garden-guide.txt | 239 characters
Basil prefers warm weather and well-drained soil. Pinch off the growing tips to encourage a fuller plant. Harvest leaves in the morning for the strongest aroma. Basil is sensitive to cold and should be protected when nighttime temperatures

Chunk 4 | Source: garden-guide.txt | 43 characters
protected when nighttime temperatures fall.

Chunk 5 | Source: compost-notes.txt | 236 characters
A healthy compost pile needs a balance of green materials, such as vegetable scraps, and brown materials, such as dry lea

## 3. Create embeddings

An embedding is a vector: a list of numbers where texts with related meaning tend to have vectors pointing in similar directions. Ollama's `/api/embed` endpoint can embed all our chunks in one request. In a full ingestion pipeline, vectors and their text/metadata are saved so they do not need to be recomputed each time.

In [9]:
chunk_texts = [chunk.page_content for chunk in chunk_documents]
embedding_result = ollama_request(
    "/api/embed",
    {"model": EMBEDDING_MODEL, "input": chunk_texts}
)
chunk_vectors = embedding_result["embeddings"]

print(f"Embedded {len(chunk_vectors)} chunks")
print(f"Embedded {len(chunk_vectors[0])} numbers")
print(f"First five values: {chunk_vectors[0][:5]}")

Embedded 8 chunks
Embedded 768 numbers
First five values: [0.033425026, 0.043426294, -0.20090681, 0.051115267, 0.004117202]


## 4. Retrieve relevant chunks

We embed the question with the same model, then compare it with each chunk vector. Cosine similarity measures how aligned two vectors are; a larger score means a closer match. `top_k` controls how many of the best matches to pass on. This tiny pure-Python search is easy to inspect; a vector database handles this efficiently at scale.

In [13]:
def consine_similarity(left, right):
    dot_product = sum(a*b for a,b in zip(left, right))
    left_length = math.sqrt(sum(value * value for value in left))
    right_length = math.sqrt(sum(value * value for value in right))
    if left_length == 0 or right_length == 0:
        return 0.0
    return dot_product/(left_length * right_length)

def retrieve(question, top_k=3):
    question_result = ollama_request(
        "/api/embed",
        {"model": EMBEDDING_MODEL, "input": [question]}
    )
    question_vector = question_result["embeddings"][0]
    scored_chunks = [
        {
            "score": consine_similarity(question_vector, vector),
            "chunk": chunk
        }
        for chunk, vector in zip(chunk_documents, chunk_vectors)
    ]
    return sorted(scored_chunks, key=lambda item: item["score"], reverse=True)[:top_k]

question = "How can I help tomato plants stay healthy?"
top_k = 3
retrived_chunk = retrieve(question, top_k=top_k)
print(f"Question: {question}")
for rank, result in enumerate(retrived_chunk, start=1):
    chunk = result["chunk"]
    print(f"Rank {rank} | Similarity {result['score']:.3f} | {chunk.metadata['source']}")
    print(chunk.page_content, end="\n\n")

Question: How can I help tomato plants stay healthy?
Rank 1 | Similarity 0.730 | garden-guide.txt
Tomatoes grow best in a sunny location that receives at least six hours of direct light each day. Water the soil deeply when the top few centimetres feel dry. Water near the soil rather than over the leaves. A layer of mulch helps the soil

Rank 2 | Similarity 0.638 | pollinator-notes.txt
Plant flowers with different bloom times to provide nectar throughout the growing season. Native plants are often well suited to local pollinators. Avoid spraying pesticides on open flowers, where bees and other pollinators may be feeding.

Rank 3 | Similarity 0.595 | garden-guide.txt
leaves. A layer of mulch helps the soil retain moisture and reduces weeds.



## 5. Generate an answer using the retrieved context

Retrieval finds candidate evidence; it does not write the answer. We combine the question and selected passages in a prompt, asking the model to stay within that evidence and say when it is missing. Grounding can reduce unsupported answers, but it is not a guarantee, so inspect the passages and verify important results.

In [14]:
def answer_with_context(question, retrived_chunks):
    context = "\n\n".join(
        f"source: {item['chunk'].metadata['source']}\n{item['chunk'].page_content}"
        for item in retrived_chunks
    )
    prompt = (
        "Answer the question using only the context below."
        "If the context does not contain the answer, say that the provided notes do not say. \n\n"
        f"Context:\n{context}\n\nQuestion: {question}\nAnswer:"
    )
    result = ollama_request(
        "/api/generate",
        {"model": LLM_MODEL, "prompt": prompt, "stream": False}
    )
    return result["response"]

answer = answer_with_context(question, retrived_chunk)
print(answer)

Water the soil deeply when the top few centimeters feel dry, and avoid spraying pesticides on the plants. A layer of mulch also helps the soil retain moisture and reduces weeds.


## 6. Experiment and recap

Try asking about compost or pollinators, then ask something absent from the notes, such as the best time to plant carrots. Rerun the retrieval cell and this answer cell. Change `top_k` to `1` or `4` and compare how much context is sent. You can also change `chunk_size` and `chunk_overlap`, then rerun the notebook from the chunking cell onward.

**Ingestion:** documents → chunks → embeddings → stored vectors and source metadata.

**Question time:** question → question embedding → similarity search → top-k chunks → prompt with context → generated answer.

The vectors in this notebook live only in memory and disappear when the kernel stops. Your project already includes Qdrant utilities; a natural next step is to store these same vectors and payloads in Qdrant instead of comparing every vector in Python.